# Bee Bright AI Training Notebook

This notebook trains AI models and generates data for the Bee Bright tutorial center:

1. **Student Recommendations**: Identifies students with grades below 75% and recommends materials by `programCategory` + `subjectItem`
2. **Tutor Recommendations**: Suggests materials per taught subject for preparing lessons
3. **Chat Rules**: Generates keyword→reply rules from CSV for the chatbot

## Where to Put
- **Notebook location**: `beebright-ui-showcase/ai_training/bee_bright_ai_training.ipynb`
- **CSV data location**: `beebright-ui-showcase/ai_training/data/`

## Step 1: Install Dependencies

Run in terminal: `pip install pandas scikit-learn`

In [1]:
import pandas as pd
import os

# Run notebook from beebright-ui-showcase/ai_training/ folder
# Data: ai_training/data/ | Output: ai_training/output/
DATA_DIR = os.path.join(os.getcwd(), 'data')
if not os.path.exists(DATA_DIR):
    DATA_DIR = 'data'
print(f'Data directory: {DATA_DIR}')

Data directory: C:\BB_4 UPDATED\beebright-ui-showcase\ai_training\data


## Step 2: Load Grades (for Student Failing Subject Detection)

**CSV columns**: studentId, tutorId, programCategory, subjectItem, score, maxScore, period, remarks

In [2]:
grades_df = pd.read_csv(os.path.join(DATA_DIR, 'grades_sample.csv'))
grades_df['pct'] = (grades_df['score'] / grades_df['maxScore'] * 100).round(0)

FAIL_PERCENT = 75
failing = grades_df[grades_df['pct'] < FAIL_PERCENT]

print('Grades loaded:')
print(grades_df)
print('\nFailing subjects (pct < 75):')
print(failing[['studentId', 'programCategory', 'subjectItem', 'score', 'maxScore', 'pct']])

Grades loaded:
  studentId   tutorId                     programCategory   subjectItem  \
0   stud001  tutor001                   Academic Tutorial   Mathematics   
1   stud001  tutor001                   Academic Tutorial       English   
2   stud001  tutor002                   Academic Tutorial       Science   
3   stud002  tutor001  Pre-Kindergarten Readiness Program       Phonics   
4   stud002  tutor001  Pre-Kindergarten Readiness Program  Numbers 1-20   
5   stud003  tutor002                   Academic Tutorial   Mathematics   

   score  maxScore   period                           remarks   pct  
0     65       100  Q1 2024                    Needs practice  65.0  
1     80       100  Q1 2024                     Good progress  80.0  
2     58       100  Q1 2024  Struggling - recommend materials  58.0  
3     72       100  Q1 2024         Slight improvement needed  72.0  
4     90       100  Q1 2024                         Excellent  90.0  
5     45       100  Q1 2024  Failing - 

## Step 3: Load Materials (for Recommendation Matching)

**CSV columns**: title, description, materialType, category, programCategory, subjectItem, assignedStudentIds

In [3]:
materials_df = pd.read_csv(os.path.join(DATA_DIR, 'materials_sample.csv'))
print('Materials loaded:')
print(materials_df)

Materials loaded:
                     title                           description materialType  \
0  Phonics Practice Sheets  Letter sounds and blending exercises          pdf   
1       Math Review Slides    Basic operations and word problems          pdf   
2      Science Video Recap         Basic science concepts for Q1        video   

        category                     programCategory  subjectItem  \
0       Practice  Pre-Kindergarten Readiness Program      Phonics   
1  Lecture Notes                   Academic Tutorial  Mathematics   
2  Video Lecture                   Academic Tutorial      Science   

  assignedStudentIds  
0    stud001,stud002  
1    stud001,stud003  
2            stud001  


## Step 4: Recommendation Logic (Student: Failing Subject → Materials)

For each failing grade, find materials where programCategory and subjectItem match, and the student is in assignedStudents.

In [4]:
recommendations = []
for _, row in failing.iterrows():
    prog = str(row['programCategory']).strip().lower()
    subj = str(row['subjectItem']).strip().lower()
    sid = str(row['studentId'])
    for _, m in materials_df.iterrows():
        m_prog = str(m.get('programCategory', '')).strip().lower()
        m_subj = str(m.get('subjectItem', '')).strip().lower()
        assigned = str(m.get('assignedStudentIds', '')).split(',')
        if prog in m_prog or m_prog in prog:
            if subj in m_subj or m_subj in subj:
                if sid in [a.strip() for a in assigned if a.strip()]:
                    recommendations.append({
                        'studentId': sid,
                        'programCategory': row['programCategory'],
                        'subjectItem': row['subjectItem'],
                        'materialTitle': m['title'],
                        'materialType': m['materialType']
                    })

rec_df = pd.DataFrame(recommendations)
print('Recommended materials for failing subjects:')
print(rec_df)

Recommended materials for failing subjects:
  studentId                     programCategory  subjectItem  \
0   stud001                   Academic Tutorial  Mathematics   
1   stud001                   Academic Tutorial      Science   
2   stud002  Pre-Kindergarten Readiness Program      Phonics   
3   stud003                   Academic Tutorial  Mathematics   

             materialTitle materialType  
0       Math Review Slides          pdf  
1      Science Video Recap        video  
2  Phonics Practice Sheets          pdf  
3       Math Review Slides          pdf  


## Step 5: Export Recommendations to JSON (for Backend)

**Where to put**: `ai_training/output/recommendation_rules.json`

In [5]:
import json

output_dir = os.path.join(DATA_DIR, '..', 'output')
os.makedirs(output_dir, exist_ok=True)

rules = {
    'failPercent': 75,
    'description': 'Student recommendation: materials for subjects where grade < failPercent',
}
with open(os.path.join(output_dir, 'recommendation_rules.json'), 'w') as f:
    json.dump(rules, f, indent=2)

print(f'Saved to {output_dir}/recommendation_rules.json')

Saved to C:\BB_4 UPDATED\beebright-ui-showcase\ai_training\data\..\output/recommendation_rules.json


## Step 6: Chat Rules from CSV

Load chat rules from `chat_rules_sample.csv` and export to JSON for the backend.

In [6]:
chat_rules_df = pd.read_csv(os.path.join(DATA_DIR, 'chat_rules_sample.csv'))

chat_rules = []
for _, row in chat_rules_df.iterrows():
    keywords = [k.strip() for k in str(row['keywords']).split(',') if k.strip()]
    chat_rules.append({'keywords': keywords, 'reply': str(row['reply'])})

with open(os.path.join(output_dir, 'chat_rules.json'), 'w') as f:
    json.dump(chat_rules, f, indent=2)

print('Chat rules exported:')
print(json.dumps(chat_rules, indent=2))

Chat rules exported:
[
  {
    "keywords": [
      "enroll",
      "enrollment",
      "join"
    ],
    "reply": "Enrollment is done through the enrollment form. After payment verification, your programs appear under My Programs."
  },
  {
    "keywords": [
      "schedule",
      "class",
      "session",
      "when"
    ],
    "reply": "Your schedule is in the Schedule tab. Sessions show date, time, and tutor."
  },
  {
    "keywords": [
      "payment",
      "pay",
      "fee",
      "gcash"
    ],
    "reply": "Payments are verified by admin. Use GCash as shown during enrollment."
  },
  {
    "keywords": [
      "struggling",
      "help",
      "recommend"
    ],
    "reply": "Use the Recommendations section in the AI tab for suggested materials per subject."
  }
]


## Step 7: Export Full Data as CSV (for Re-import)

Export processed grades and recommendations to CSV for backup/training.

In [7]:
grades_df.to_csv(os.path.join(output_dir, 'grades_with_pct.csv'), index=False)
if len(recommendations) > 0:
    rec_df.to_csv(os.path.join(output_dir, 'student_recommendations.csv'), index=False)

print(f'Exported to {output_dir}/')

Exported to C:\BB_4 UPDATED\beebright-ui-showcase\ai_training\data\..\output/
